# CMIP6 data prep for DSC 106 Project 3

This notebook turns the huge CMIP6 climate archive into one small CSV that D3 can load from GitHub Pages.

**What it does**
1. Reads the public CMIP6 catalog (a CSV listing every dataset on Google Cloud).
2. Picks a few climate models and four scenarios: `historical`, `ssp126` (low emissions), `ssp245` (middle), `ssp585` (high).
3. Streams only the near-surface air temperature (`tas`, monthly) for those runs. No login is needed.
4. Computes an area-weighted global average for each year.
5. Converts to Celsius and to an anomaly relative to a 1850-1900 baseline.
6. Exports `data/global_tas.csv` and `data/model_runs.csv`.

**Heads up:** loading takes a few minutes the first time. Each result is cached in `data/cache/`, so re-running is fast.

## 1. Setup
Run this once in Colab. If you get a zarr or xarray error later, try `pip install "zarr<3"` and restart the runtime.

In [ ]:
!pip install -q gcsfs zarr xarray pandas matplotlib cftime

In [ ]:
import os
import re

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

CATALOG_URL = "https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv"

VARIABLE = "tas"      # near-surface air temperature
TABLE = "Amon"        # monthly atmosphere data
EXPERIMENTS = ["historical", "ssp126", "ssp245", "ssp585"]

# Models to try. We keep the first N_MODELS that have all four experiments.
CANDIDATE_MODELS = [
    "CESM2", "UKESM1-0-LL", "MPI-ESM1-2-HR", "CanESM5",
    "GFDL-ESM4", "IPSL-CM6A-LR", "MRI-ESM2-0", "NorESM2-LM",
]
N_MODELS = 5

BASELINE = (1850, 1900)   # anomalies are relative to this period
LAST_YEAR = 2100

OUT_DIR = "data"
CACHE_DIR = os.path.join(OUT_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

## 2. Read the catalog
This is a big CSV (tens of MB), so it takes a moment. Each row points to one Zarr store in `zstore`.

In [ ]:
df = pd.read_csv(CATALOG_URL)
print(df.shape)
print(list(df.columns))
df.head(3)

## 3. Pick the runs
For each model we need one **member** (an ensemble run, like `r1i1p1f1`) that exists in **all four** experiments, so the lines are comparable. We prefer the lowest-numbered member and the standard `gn` grid.

In [ ]:
cat = df[
    (df.table_id == TABLE)
    & (df.variable_id == VARIABLE)
    & (df.experiment_id.isin(EXPERIMENTS))
    & (df.source_id.isin(CANDIDATE_MODELS))
].copy()

# One row per (model, experiment, member): prefer grid "gn", then newest version.
cat["grid_rank"] = (cat.grid_label != "gn").astype(int)
cat["version"] = cat["version"].astype(str)
cat = cat.sort_values(
    ["source_id", "experiment_id", "member_id", "grid_rank", "version"],
    ascending=[True, True, True, True, False],
).drop_duplicates(["source_id", "experiment_id", "member_id"])


def member_key(member):
    """r1i1p1f1 -> (1, 1, 1, 1) so members sort numerically."""
    return tuple(int(x) for x in re.findall(r"\d+", member))


def preferred_member(sub):
    common = set.intersection(*[set(sub[sub.experiment_id == e].member_id) for e in EXPERIMENTS])
    return min(common, key=member_key) if common else None


rows = []
for model in sorted(cat.source_id.unique()):
    sub = cat[cat.source_id == model]
    member = preferred_member(sub)
    if member is None:
        print(f"skipping {model}: no member with all experiments")
        continue
    rows.append(sub[sub.member_id == member])
    if len({r.source_id.iloc[0] for r in rows}) >= N_MODELS:
        break

runs = pd.concat(rows)[["source_id", "experiment_id", "member_id", "grid_label", "zstore"]]
runs = runs.rename(columns={"source_id": "model", "experiment_id": "scenario"}).reset_index(drop=True)
print(runs.groupby("model").member_id.first())
runs

## 4. Load one run and take a global annual mean
`annual_global_mean` opens a Zarr store lazily, averages over latitude and longitude, then over the months in each year. Only the small result is downloaded and computed.

The latitude average is **area-weighted** (weight = cosine of latitude). Without that, the small polar grid cells would count as much as the big tropical ones.

In [ ]:
def annual_global_mean(zstore):
    ds = xr.open_zarr(zstore, consolidated=True, storage_options={"token": "anon"})
    ds = ds.rename({k: v for k, v in {"latitude": "lat", "longitude": "lon"}.items() if k in ds.dims})
    weights = np.cos(np.deg2rad(ds["lat"]))
    global_mean = ds[VARIABLE].weighted(weights).mean(dim=["lat", "lon"])
    annual = global_mean.groupby("time.year").mean("time").squeeze(drop=True)
    out = annual.load().to_series().rename("tas_k").reset_index()
    return out[["year", "tas_k"]]


# Quick test on a single run (the first row).
test = annual_global_mean(runs.zstore.iloc[0])
print(runs.iloc[0][["model", "scenario", "member_id"]].to_dict())
test.head()

## 5. Load every run (cached)
This loops over all model and scenario combinations. Each result is saved to `data/cache/` so a crash or a re-run does not start over.

In [ ]:
frames = []
for i, r in runs.iterrows():
    cache_file = os.path.join(CACHE_DIR, f"{r.model}_{r.scenario}_{r.member_id}.csv")
    if os.path.exists(cache_file):
        part = pd.read_csv(cache_file)
    else:
        print(f"[{i + 1}/{len(runs)}] loading {r.model} {r.scenario} {r.member_id} ...")
        try:
            part = annual_global_mean(r.zstore)
        except Exception as e:
            print(f"   FAILED: {e}")
            continue
        part.to_csv(cache_file, index=False)
    part = part.assign(model=r.model, scenario=r.scenario, member_id=r.member_id)
    frames.append(part)

raw = pd.concat(frames, ignore_index=True)
raw.groupby(["model", "scenario"]).year.agg(["min", "max", "count"])

## 6. Clean up and compute anomalies
- Kelvin to Celsius.
- Keep years up to 2100 (some scenarios run to 2300).
- Anomaly = temperature minus that model's own 1850-1900 average from the `historical` run. Using each model's own baseline removes differences in how warm each model runs on average, so the lines show *change*.

In [ ]:
data = raw[raw.year <= LAST_YEAR].copy()
data["tas_c"] = data.tas_k - 273.15

base = (
    data[(data.scenario == "historical") & data.year.between(*BASELINE)]
    .groupby("model").tas_c.mean()
    .rename("baseline_c")
)
data = data.join(base, on="model")
data["anomaly_c"] = data.tas_c - data.baseline_c

data = data[["model", "scenario", "member_id", "year", "tas_c", "anomaly_c"]]
data = data.sort_values(["model", "scenario", "year"]).reset_index(drop=True)
data[["tas_c", "anomaly_c"]] = data[["tas_c", "anomaly_c"]].round(3)

print(base.round(2))
data.head()

## 7. Sanity check
Quick static plot to confirm the data looks right: all scenarios should track together through 2014, then fan out, with `ssp585` warming the most.

For plotting, each future scenario is joined to the historical record so the lines are continuous.

In [ ]:
colors = {"ssp126": "tab:green", "ssp245": "tab:orange", "ssp585": "tab:red"}
hist = data[data.scenario == "historical"]

fig, ax = plt.subplots(figsize=(10, 5))
for scen, color in colors.items():
    fut = data[data.scenario == scen]
    for model in fut.model.unique():
        line = pd.concat([hist[hist.model == model], fut[fut.model == model]])
        ax.plot(line.year, line.anomaly_c, color=color, alpha=0.25, lw=0.8)
    mean_line = (
        pd.concat([hist, fut]).groupby("year").anomaly_c.mean()
    )
    ax.plot(mean_line.index, mean_line.values, color=color, lw=2.5, label=f"{scen} (model mean)")

ax.axvline(2014.5, color="gray", ls=":")
ax.set_ylabel("Global temperature anomaly (C, vs 1850-1900)")
ax.set_xlabel("Year")
ax.set_title("Global mean near-surface temperature, CMIP6")
ax.legend()
plt.show()

## 8. Export for the website
Two files go into your repo's `data/` folder:
- `global_tas.csv`: one row per model, scenario, and year (what D3 loads).
- `model_runs.csv`: which exact runs you used, handy for your write-up.

Check the file size. A few hundred KB is perfect for GitHub Pages.

In [ ]:
data.to_csv(os.path.join(OUT_DIR, "global_tas.csv"), index=False)
runs.to_csv(os.path.join(OUT_DIR, "model_runs.csv"), index=False)

size_kb = os.path.getsize(os.path.join(OUT_DIR, "global_tas.csv")) / 1024
print(f"global_tas.csv: {len(data)} rows, {size_kb:.0f} KB")
print(data.groupby(["model", "scenario"]).size().unstack())

## 9. Using it in D3
```js
const rows = await d3.csv("data/global_tas.csv", d3.autoType);
```
Columns: `model, scenario, member_id, year, tas_c, anomaly_c`.

**For your write-up, remember to describe these transformations** (the rubric asks for them to be stated clearly, e.g. in the plot subtitle):
- monthly values averaged to annual means
- area-weighted global average
- Celsius anomaly versus each model's 1850-1900 average
- one ensemble member per model; years capped at 2100

Download `data/global_tas.csv` and `data/model_runs.csv` from Colab (folder icon, right-click, Download) and commit them to your repo.